## 01c_ingest_quality_data — RCM Intelligence Platform
### Purpose
Ingests CMS Hospital Compare quality and patient experience
data from public CMS APIs into Bronze layer.

### Data sources
1. Hospital General Information — overall ratings, ownership
2. Hospital Quality Measures — mortality, readmissions, safety
3. HCAHPS Patient Satisfaction — patient experience scores

### Why this matters
Adds clinical quality and patient experience dimensions
to the platform. Combined with RCM financial data this
creates a true 360-degree hospital performance view.

### Outputs
- rcm_platform.rcm_bronze.hospital_quality_raw
- rcm_platform.rcm_bronze.hospital_hcahps_raw

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Bronze |
| Run order  | 3 — after 01a and 01b |
| Depends on | 00_setup_databases |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
import requests
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql.functions import lit, col
from pyspark.sql.types import StringType

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "01c_ingest_quality_data"

TBL_QUALITY = f"{BRONZE_DB}.hospital_quality_raw"
TBL_HCAHPS  = f"{BRONZE_DB}.hospital_hcahps_raw"

print(f"Batch ID : {BATCH_ID}")
print(f"Outputs  :")
print(f"  {TBL_QUALITY}")
print(f"  {TBL_HCAHPS}")

In [0]:
# ============================================================
# STEP 1 — INGEST HOSPITAL QUALITY MEASURES
# CMS Hospital Compare — mortality, readmissions, safety
# ============================================================

print("=" * 55)
print("  INGESTING HOSPITAL QUALITY MEASURES")
print("=" * 55)

# Two quality datasets — general info + quality measures
QUALITY_GENERAL_URL = "https://data.cms.gov/provider-data/api/1/datastore/query/yc9t-dgbk/0/download?format=csv"

QUALITY_MEASURES_URL = "https://data.cms.gov/provider-data/api/1/datastore/query/ynj2-r877/0/download?format=csv"

print(f"Fetching quality measures from: {QUALITY_MEASURES_URL}")

try:
    response = requests.get(QUALITY_MEASURES_URL, timeout=120)
    response.raise_for_status()
    
    from io import StringIO
    df_quality_pd = pd.read_csv(StringIO(response.text), dtype=str)
    
    print(f"Downloaded rows    : {len(df_quality_pd):,}")
    print(f"Downloaded columns : {len(df_quality_pd.columns)}")
    print(f"\nColumn names:")
    for c in df_quality_pd.columns:
        print(f"  {c}")

except Exception as e:
    print(f"Error: {e}")
    # Try backup URL
    print("Trying backup URL...")
    BACKUP_URL = "https://data.cms.gov/provider-data/api/1/datastore/query/xubh-q36u/0/download?format=csv"
    response = requests.get(BACKUP_URL, timeout=120)
    df_quality_pd = pd.read_csv(StringIO(response.text), dtype=str)
    print(f"Backup rows    : {len(df_quality_pd):,}")
    print(f"Backup columns : {len(df_quality_pd.columns)}")
    for c in df_quality_pd.columns:
        print(f"  {c}")

In [0]:
# ============================================================
# STEP 3 — INGEST HCAHPS PATIENT SATISFACTION
# CMS Patient Experience Survey Results
# ============================================================

print("=" * 55)
print("  INGESTING HCAHPS PATIENT SATISFACTION")
print("=" * 55)

HCAHPS_URL = "https://data.cms.gov/provider-data/api/1/datastore/query/dgck-syfz/0/download?format=csv"

print(f"Fetching from: {HCAHPS_URL}")

try:
    response = requests.get(HCAHPS_URL, timeout=120)
    response.raise_for_status()
    
    df_hcahps_pd = pd.read_csv(StringIO(response.text), dtype=str)
    
    print(f"Downloaded rows    : {len(df_hcahps_pd):,}")
    print(f"Downloaded columns : {len(df_hcahps_pd.columns)}")
    print(f"\nColumn names:")
    for c in df_hcahps_pd.columns:
        print(f"  {c}")

except Exception as e:
    print(f"Error fetching HCAHPS data: {e}")
    raise

In [0]:
# ============================================================
# STEP 3 — INGEST READMISSIONS DATA (HRRP)
# ============================================================

print("=" * 55)
print("  INGESTING READMISSIONS DATA")
print("=" * 55)

READM_URL = "https://data.cms.gov/provider-data/api/1/datastore/query/9n3s-kdb3/0/download?format=csv"

TBL_READM = f"{BRONZE_DB}.hospital_readmissions_raw"

print(f"Fetching from: {READM_URL}")

try:
    response = requests.get(READM_URL, timeout=120)
    response.raise_for_status()
    df_readm_pd = pd.read_csv(StringIO(response.text), dtype=str)

    print(f"Downloaded rows    : {len(df_readm_pd):,}")
    print(f"Downloaded columns : {len(df_readm_pd.columns)}")

    # Clean column names
    df_readm_pd.columns = [
        c.strip()
         .lower()
         .replace(" ", "_")
         .replace("/", "_")
         .replace("-", "_")
         .replace("(", "")
         .replace(")", "")
        for c in df_readm_pd.columns
    ]

    print(f"\nCleaned columns: {df_readm_pd.columns.tolist()}")

    # Convert to Spark and write
    df_readm_spark = spark.createDataFrame(df_readm_pd.astype(str)) \
        .withColumn("_source",           lit("cms_hrrp_readmissions")) \
        .withColumn("_batch_id",         lit(BATCH_ID)) \
        .withColumn("_batch_date",       lit(BATCH_DATE)) \
        .withColumn("_ingested_at",      lit(BATCH_TIMESTAMP)) \
        .withColumn("_pipeline_name",    lit(PIPELINE_NAME)) \
        .withColumn("_pipeline_version", lit(PIPELINE_VERSION))

    df_readm_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TBL_READM)

    readm_rows = spark.table(TBL_READM).count()
    print(f"\nRows written : {readm_rows:,}")
    print(f"Table        : {TBL_READM}")

    display(spark.table(TBL_READM).limit(5))

except Exception as e:
    print(f"Error: {e}")
    raise

In [0]:
# ============================================================
# STEP 2 — WRITE QUALITY TO BRONZE
# ============================================================

print("=" * 55)
print("  WRITING QUALITY DATA TO BRONZE")
print("=" * 55)

# Clean column names
df_quality_pd.columns = [
    c.strip()
     .lower()
     .replace(" ", "_")
     .replace("/", "_")
     .replace("-", "_")
     .replace("(", "")
     .replace(")", "")
     .replace(".", "")
    for c in df_quality_pd.columns
]

print(f"Cleaned column names:")
for c in df_quality_pd.columns:
    print(f"  {c}")

# Convert to Spark
df_quality_spark = spark.createDataFrame(df_quality_pd.astype(str)) \
    .withColumn("_source",          lit("cms_hospital_compare")) \
    .withColumn("_batch_id",        lit(BATCH_ID)) \
    .withColumn("_batch_date",      lit(BATCH_DATE)) \
    .withColumn("_ingested_at",     lit(BATCH_TIMESTAMP)) \
    .withColumn("_pipeline_name",   lit(PIPELINE_NAME)) \
    .withColumn("_pipeline_version",lit(PIPELINE_VERSION))

# Write
df_quality_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_QUALITY)

quality_rows = spark.table(TBL_QUALITY).count()
print(f"\nRows written : {quality_rows:,}")
print(f"Table        : {TBL_QUALITY}")

display(spark.table(TBL_QUALITY).limit(5))

In [0]:
# ============================================================
# STEP 4 — WRITE HCAHPS TO BRONZE
# ============================================================

print("=" * 55)
print("  WRITING HCAHPS DATA TO BRONZE")
print("=" * 55)

# Clean column names
df_hcahps_pd.columns = [
    c.strip()
     .lower()
     .replace(" ", "_")
     .replace("/", "_")
     .replace("-", "_")
     .replace("(", "")
     .replace(")", "")
     .replace(".", "")
    for c in df_hcahps_pd.columns
]

print(f"Cleaned column names:")
for c in df_hcahps_pd.columns:
    print(f"  {c}")

# Convert to Spark
df_hcahps_spark = spark.createDataFrame(df_hcahps_pd.astype(str)) \
    .withColumn("_source",           lit("cms_hcahps")) \
    .withColumn("_batch_id",         lit(BATCH_ID)) \
    .withColumn("_batch_date",        lit(BATCH_DATE)) \
    .withColumn("_ingested_at",       lit(BATCH_TIMESTAMP)) \
    .withColumn("_pipeline_name",     lit(PIPELINE_NAME)) \
    .withColumn("_pipeline_version",  lit(PIPELINE_VERSION))

# Write
df_hcahps_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_HCAHPS)

hcahps_rows = spark.table(TBL_HCAHPS).count()
print(f"\nRows written : {hcahps_rows:,}")
print(f"Table        : {TBL_HCAHPS}")

display(spark.table(TBL_HCAHPS).limit(5))

In [0]:
log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "bronze",
    status           = "SUCCESS",
    rows_read        = quality_rows + hcahps_rows + readm_rows,
    rows_written     = quality_rows + hcahps_rows + readm_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"hospital quality: {quality_rows:,} rows | "
        f"HCAHPS: {hcahps_rows:,} rows | "
        f"readmissions HRRP: {readm_rows:,} rows"
    )
)
print(f"\nSummary:")
print(f"  Hospital quality rows : {quality_rows:,}")
print(f"  HCAHPS rows           : {hcahps_rows:,}")
print(f"  Readmissions rows     : {readm_rows:,}")
print(f"  Total ingested        : {quality_rows + hcahps_rows + readm_rows:,}")